In [ ]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_anchor_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': '77fbfdd8018dc48175b708232bc0f4efb2d4da81',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'stage_anchor_continue','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


# 난이도별 Stage ACT 앵커 + 성공 보상 RL

Easy 100% `block_03.pt`, Medium 40.625% `block_02.pt`, Hard 4.167% `best_val.pt`를 SHA-256으로 확인해 복원합니다. Easy는 동결합니다. Medium과 Hard는 서로 다른 모델·optimizer·결과 폴더를 사용합니다.

추가 학습은 demonstration action을 정답으로 두는 imitation loss를 사용하지 않습니다. 실제 WarehouseSort 환경의 sparse reward, 즉 `success_count`가 증가할 때의 보상만으로 PPO 업데이트합니다. 기존 앵커의 encoder·stage/gate 판단은 동결하고 action decoder만 작게 업데이트하며, 앵커와의 KL 제약으로 급격한 붕괴를 막습니다.

성능 선택은 loss가 아니라 공식 `eval.py`의 `SORT ACCURACY`로 합니다. RL 결과가 같은 seed의 앵커 점수를 **엄격히 초과할 때만** 채택하며 동점과 하락은 앵커를 유지합니다. Drive는 사용하지 않습니다.


In [ ]:
# 06 · 검증 앵커 복원 / 공식 평가 함수
import json, os, re, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Video, display
from stage_anchor_continue import (ANCHORS, package, prepare, prepare_success_rl,
                                   train_success_rl)

MAX_STEPS = 200
UPSTREAM = Path(CFG['repo_dir'])
OFFICIAL = UPSTREAM/'conf/eval/default.yaml'
anchor_exp = prepare(experiment, run_suffix='_anchor_success_base_v1')
RUN_DIR = Path(anchor_exp.run_dir)
BASELINE = package(anchor_exp, folder_name='anchor_candidate')

def run_official(candidate, level, label):
    output = RUN_DIR/level/'success_rl_official_eval'/candidate.name/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_policy:load_policy',
        'checkpoint='+str(candidate/'checkpoints'/level/'model.pt'),
        'eval_config='+str(OFFICIAL), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(candidate)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line); handle.flush(); print(line, end='', flush=True)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            process.stdout.close()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    match = re.search(r'SORT ACCURACY:\s+([0-9.]+)\s*%', log.read_text(encoding='utf-8'))
    if not match:
        raise ValueError(f'공식 점수를 읽지 못했습니다: {log}')
    videos = sorted((output/'videos').rglob('*.mp4'), key=lambda p:p.stat().st_mtime)
    if videos:
        display(Video(str(videos[-1]), embed=True, width=900))
    return {'level':level, 'score':float(match.group(1))/100, 'log':str(log),
            'checkpoint':str(candidate/'checkpoints'/level/'model.pt')}


In [ ]:
# 07 · 학습 전 공식 default test + 영상
BASELINE_RESULTS = {level:run_official(BASELINE, level, 'anchor_default')
                    for level in ('easy','medium','hard')}
print(json.dumps(BASELINE_RESULTS, indent=2))


## 성공 보상 RL

각 RL iteration은 무작위 실전 환경 16개를 200 step 실행합니다. 보상은 환경이 제공하는 `delta success_count`뿐입니다. 성공 상자가 하나도 없는 rollout은 업데이트를 건너뜁니다. 매 iteration마다 모델·optimizer·난수 상태를 저장하므로 셀 재실행 시 이어집니다.


In [ ]:
# 09 · Medium 성공 보상 RL → 같은 공식 test
medium_rl = prepare_success_rl(anchor_exp, 'medium', iterations=8, num_envs=16,
                               lr=5e-6, xyz_std=.05)
train_success_rl(medium_rl, 'medium')
MEDIUM_RL = package(medium_rl, use_trained={'medium':True,'hard':False},
                    folder_name='medium_success_rl_candidate')
MEDIUM_RESULT = run_official(MEDIUM_RL, 'medium', 'success_rl_default')
MEDIUM_USE_RL = MEDIUM_RESULT['score'] > BASELINE_RESULTS['medium']['score']
print('Medium 선택:', 'success RL' if MEDIUM_USE_RL else '검증 앵커',
      MEDIUM_RESULT['score'], 'vs', BASELINE_RESULTS['medium']['score'])


In [ ]:
# 10 · Hard 성공 보상 RL → 같은 공식 test
hard_rl = prepare_success_rl(anchor_exp, 'hard', iterations=10, num_envs=16,
                             lr=5e-6, xyz_std=.06)
train_success_rl(hard_rl, 'hard')
HARD_RL = package(hard_rl, use_trained={'medium':False,'hard':True},
                  folder_name='hard_success_rl_candidate')
HARD_RESULT = run_official(HARD_RL, 'hard', 'success_rl_default')
HARD_USE_RL = HARD_RESULT['score'] > BASELINE_RESULTS['hard']['score']
print('Hard 선택:', 'success RL' if HARD_USE_RL else '검증 앵커',
      HARD_RESULT['score'], 'vs', BASELINE_RESULTS['hard']['score'])


In [ ]:
# 11 · 공식 점수로 자동 선택한 단일 제출 candidate + 최종 재검증 + ZIP
selected = {}
if globals().get('MEDIUM_USE_RL', False):
    selected['medium'] = Path(MEDIUM_RESULT['checkpoint'])
if globals().get('HARD_USE_RL', False):
    selected['hard'] = Path(HARD_RESULT['checkpoint'])
FINAL = package(anchor_exp, checkpoint_overrides=selected, folder_name='final_success_rl_candidate')
FINAL_RESULTS = {level:run_official(FINAL, level, 'final_default')
                 for level in ('easy','medium','hard')}
print(json.dumps({'selected':{k:str(v) for k,v in selected.items()},
                  'official_results':FINAL_RESULTS}, indent=2))
archive = shutil.make_archive(str(RUN_DIR/'stage_act_success_rl_submission'), 'zip', root_dir=FINAL)
from google.colab import files
files.download(archive)
